# Land Cover Classification — Train on EuroSAT (RGB)

Run this notebook in Google Colab with a GPU runtime:
`Runtime -> Change runtime type -> T4 GPU`

**No pip installs needed** — this version downloads the EuroSAT dataset
directly as a zip file and loads it with `tf.keras.utils.image_dataset_from_directory`,
which avoids `tensorflow_datasets` and the protobuf version conflicts it causes
in Colab.

At the end you'll download `landcover_cnn.h5` and put it in your project's `model/` folder.

In [ ]:
import os
import zipfile
import urllib.request
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 15
SEED = 42
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Download and extract EuroSAT (RGB) directly — no tensorflow_datasets needed.
# Try a couple of mirrors in case one is slow/unreachable from Colab's network.
ZIP_PATH = "EuroSAT.zip"
EXTRACT_ROOT = "eurosat_data"
MIRRORS = [
    "https://huggingface.co/datasets/torchgeo/eurosat/resolve/main/EuroSAT.zip",
    "https://madm.dfki.de/files/sentinel/EuroSAT.zip",
]

if not os.path.exists(EXTRACT_ROOT):
    last_error = None
    for url in MIRRORS:
        try:
            print(f"Trying {url} ...")
            urllib.request.urlretrieve(url, ZIP_PATH)
            print("Download succeeded.")
            last_error = None
            break
        except Exception as e:
            print(f"Failed: {e}")
            last_error = e
    if last_error is not None:
        raise RuntimeError(
            "Could not download EuroSAT from any mirror. Re-run this cell "
            "(transient network issues are common in Colab), or try again "
            "in a few minutes."
        ) from last_error
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_ROOT)
    print("Done.")
else:
    print("Dataset already downloaded.")

# The zip extracts into a folder called '2750' containing one subfolder per class
DATA_DIR = os.path.join(EXTRACT_ROOT, "2750")
print("Classes found:", sorted(os.listdir(DATA_DIR)))

In [ ]:
# Build train/val/test datasets straight from the image folders.
# Keras splits off 20% for validation+test here; we then split that in half below.
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

val_test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

CLASS_NAMES = train_ds_raw.class_names
print("Class order:", CLASS_NAMES)

# Split the held-out 20% evenly into validation and test
val_test_batches = tf.data.experimental.cardinality(val_test_ds_raw)
test_ds_raw = val_test_ds_raw.take(val_test_batches // 2)
val_ds_raw = val_test_ds_raw.skip(val_test_batches // 2)

In [ ]:
def normalize(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (train_ds_raw.map(normalize, num_parallel_calls=AUTOTUNE)
            .map(augment, num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = val_ds_raw.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds_raw.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=90); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
fig.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
os.makedirs("model", exist_ok=True)
model.save("model/landcover_cnn.h5")
print("Saved. Now download model/landcover_cnn.h5 from the Colab file browser (left sidebar)\nand place it in your local project's model/ folder before deploying.")